# Phase 4: Feature Engineering

## 🎯 Learning Objectives

By the end of this notebook, you will:

- ✅ Handle missing data effectively
- ✅ Detect and treat outliers
- ✅ Encode categorical variables
- ✅ Scale and normalize features
- ✅ Create new features from existing ones
- ✅ Select the most important features

**Time Required:** 1-2 weeks  
**Difficulty:** Intermediate to Advanced  
**Prerequisites:** Phases 0-3 completed

---

**"Data preparation is 80% of the work in ML!"**

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   LabelEncoder, OneHotEncoder, OrdinalEncoder)
from sklearn.impute import SimpleImputer, KNNImputer

# Feature Selection
from sklearn.feature_selection import (SelectKBest, f_classif, mutual_info_classif,
                                       RFE, SelectFromModel)

# Models for feature importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Train-test split
from sklearn.model_selection import train_test_split

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ All libraries imported successfully!")

---

# Part A: Handling Missing Data

## 4.1 Identifying Missing Values

In [ ]:
# Create sample dataset with missing values

np.random.seed(42)
n = 200

data = pd.DataFrame({
    'Age': np.random.randint(18, 65, n),
    'Income': np.random.normal(55000, 20000, n),
    'Experience': np.random.randint(0, 40, n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'Department': np.random.choice(['Sales', 'Tech', 'HR', 'Finance'], n),
    'Satisfaction': np.random.randint(1, 11, n),
    'Performance': np.random.choice(['Low', 'Medium', 'High'], n),
    'Target': np.random.randint(0, 2, n)
})

# Introduce missing values
missing_idx = np.random.choice(n, 30, replace=False)
data.loc[missing_idx[:10], 'Age'] = np.nan
data.loc[missing_idx[10:20], 'Income'] = np.nan
data.loc[missing_idx[20:25], 'Education'] = np.nan
data.loc[missing_idx[25:], 'Satisfaction'] = np.nan

print("Dataset with Missing Values:")
print(data.head(15))

In [ ]:
# Check missing values

print("=== Missing Values Analysis ===")

# Count missing
missing_count = data.isnull().sum()
missing_percent = (data.isnull().sum() / len(data)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent
})

print(missing_df[missing_df['Missing Count'] > 0])

# Visualize
plt.figure(figsize=(10, 6))
sns.heatmap(data.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd')
plt.title('Missing Values Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.2 Handling Missing Values: Different Strategies

In [ ]:
# Strategy 1: Drop rows with missing values

df_dropped = data.dropna()

print(f"Original shape: {data.shape}")
print(f"After dropping: {df_dropped.shape}")
print(f"Rows lost: {len(data) - len(df_dropped)} ({(len(data) - len(df_dropped))/len(data)*100:.1f}%)")

print("\n⚠️ Warning: Only use when missing data is small (<5%)")

In [ ]:
# Strategy 2: Simple Imputation (Mean/Median/Mode)

df_imputed = data.copy()

# Numerical columns: Impute with median (robust to outliers)
num_cols = ['Age', 'Income', 'Satisfaction']
for col in num_cols:
    df_imputed[col].fillna(df_imputed[col].median(), inplace=True)

# Categorical columns: Impute with mode (most frequent)
cat_cols = ['Education']
for col in cat_cols:
    df_imputed[col].fillna(df_imputed[col].mode()[0], inplace=True)

print("After Simple Imputation:")
print(f"Missing values: {df_imputed.isnull().sum().sum()}")

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, num_cols):
    # Original (without NaN)
    ax.hist(data[col].dropna(), bins=20, alpha=0.5, label='Original', density=True)
    # Imputed
    ax.hist(df_imputed[col], bins=20, alpha=0.5, label='Imputed', density=True)
    ax.axvline(data[col].median(), color='red', linestyle='--', label='Median')
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribution Before/After Median Imputation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Strategy 3: Using sklearn SimpleImputer

df_sklearn = data.copy()

# Numerical imputer
num_imputer = SimpleImputer(strategy='median')
df_sklearn[num_cols] = num_imputer.fit_transform(df_sklearn[num_cols])

# Categorical imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df_sklearn[cat_cols] = cat_imputer.fit_transform(df_sklearn[cat_cols])

print("Using sklearn SimpleImputer:")
print(f"Missing values: {df_sklearn.isnull().sum().sum()}")

print("\nImputation strategies available:")
print("- 'mean': Replace with column mean")
print("- 'median': Replace with column median (robust)")
print("- 'most_frequent': Replace with mode")
print("- 'constant': Replace with a constant value")

In [ ]:
# Strategy 4: KNN Imputation (Advanced)

df_knn = data.copy()

# First encode categoricals for KNN
df_knn_encoded = df_knn.copy()
for col in ['Education', 'Department', 'Performance']:
    df_knn_encoded[col] = LabelEncoder().fit_transform(df_knn_encoded[col].fillna('Unknown'))

# Apply KNN Imputer on numerical + encoded columns
knn_cols = ['Age', 'Income', 'Experience', 'Satisfaction']
knn_imputer = KNNImputer(n_neighbors=5)
df_knn[knn_cols] = knn_imputer.fit_transform(df_knn[knn_cols])

# Still need to handle categorical missing
df_knn['Education'].fillna(df_knn['Education'].mode()[0], inplace=True)

print("After KNN Imputation:")
print(f"Missing values: {df_knn.isnull().sum().sum()}")

print("\n✅ KNN Imputation uses similar rows to fill missing values")
print("   Better for data with correlations between features")

---

# Part B: Handling Outliers

## 4.3 Detecting Outliers

In [ ]:
# Create data with outliers

np.random.seed(42)
n = 200

# Normal data
salaries = np.random.normal(50000, 15000, n)
ages = np.random.normal(35, 10, n)

# Add outliers
salaries = np.append(salaries, [200000, 250000, 300000, 350000])  # High salary outliers
ages = np.append(ages, [95, 100, 5, 3])  # Age outliers

df_outliers = pd.DataFrame({
    'Salary': salaries,
    'Age': ages
})

print("Dataset with Outliers:")
print(df_outliers.describe())

In [ ]:
# Visualize outliers with box plots

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Salary boxplot
axes[0].boxplot(df_outliers['Salary'])
axes[0].set_ylabel('Salary ($)', fontsize=12)
axes[0].set_title('Salary Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Age boxplot
axes[1].boxplot(df_outliers['Age'])
axes[1].set_ylabel('Age', fontsize=12)
axes[1].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Detecting Outliers with Box Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Points beyond whiskers are potential outliers")

In [ ]:
# Method 1: IQR (Interquartile Range) Method

def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return outliers, lower_bound, upper_bound

# Apply to Salary
outliers_salary, lb_sal, ub_sal = detect_outliers_iqr(df_outliers, 'Salary')
print(f"Salary Outliers (IQR Method):")
print(f"  Lower Bound: ${lb_sal:,.0f}")
print(f"  Upper Bound: ${ub_sal:,.0f}")
print(f"  Outliers Found: {len(outliers_salary)}")

# Apply to Age
outliers_age, lb_age, ub_age = detect_outliers_iqr(df_outliers, 'Age')
print(f"\nAge Outliers (IQR Method):")
print(f"  Lower Bound: {lb_age:.1f}")
print(f"  Upper Bound: {ub_age:.1f}")
print(f"  Outliers Found: {len(outliers_age)}")

In [ ]:
# Method 2: Z-Score Method

from scipy import stats

def detect_outliers_zscore(data, column, threshold=3):
    """Detect outliers using Z-Score method"""
    z_scores = np.abs(stats.zscore(data[column]))
    outliers = data[z_scores > threshold]
    return outliers, z_scores

# Apply to Salary
outliers_salary_z, z_scores = detect_outliers_zscore(df_outliers, 'Salary')
print(f"Salary Outliers (Z-Score > 3):")
print(f"  Outliers Found: {len(outliers_salary_z)}")

# Visualize Z-scores
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(range(len(z_scores)), z_scores, alpha=0.5)
plt.axhline(y=3, color='r', linestyle='--', label='Threshold (z=3)')
plt.xlabel('Index')
plt.ylabel('Z-Score')
plt.title('Salary Z-Scores', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(z_scores, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(x=3, color='r', linestyle='--', label='Threshold')
plt.xlabel('Z-Score')
plt.ylabel('Frequency')
plt.title('Distribution of Z-Scores', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Handling Outliers: Different Strategies

df_clean = df_outliers.copy()

# Strategy 1: Remove outliers
Q1 = df_clean['Salary'].quantile(0.25)
Q3 = df_clean['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_removed = df_clean[(df_clean['Salary'] >= lower_bound) & (df_clean['Salary'] <= upper_bound)]
print(f"Strategy 1 - Removed Outliers:")
print(f"  Original: {len(df_clean)} rows")
print(f"  After: {len(df_removed)} rows")

# Strategy 2: Cap outliers (Winsorization)
df_capped = df_clean.copy()
df_capped['Salary_Capped'] = df_capped['Salary'].clip(lower=lower_bound, upper=upper_bound)
print(f"\nStrategy 2 - Capped Outliers:")
print(f"  Max before: ${df_clean['Salary'].max():,.0f}")
print(f"  Max after: ${df_capped['Salary_Capped'].max():,.0f}")

# Strategy 3: Transform (log transformation)
df_clean['Salary_Log'] = np.log1p(df_clean['Salary'])
print(f"\nStrategy 3 - Log Transformation:")
print(f"  Reduces impact of extreme values")

In [ ]:
# Visualize the effect of handling outliers

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
axes[0, 0].hist(df_outliers['Salary'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Original Data', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Salary')
axes[0, 0].grid(True, alpha=0.3)

# After removing
axes[0, 1].hist(df_removed['Salary'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('After Removing Outliers', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Salary')
axes[0, 1].grid(True, alpha=0.3)

# After capping
axes[1, 0].hist(df_capped['Salary_Capped'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_title('After Capping Outliers', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Salary')
axes[1, 0].grid(True, alpha=0.3)

# After log transform
axes[1, 1].hist(df_clean['Salary_Log'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title('After Log Transform', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Log(Salary)')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Different Strategies for Handling Outliers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

# Part C: Encoding Categorical Variables

## 4.4 Different Encoding Techniques

In [ ]:
# Create sample data with categorical variables

np.random.seed(42)
n = 100

df_cat = pd.DataFrame({
    'Color': np.random.choice(['Red', 'Blue', 'Green'], n),
    'Size': np.random.choice(['Small', 'Medium', 'Large'], n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'City': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], n),
    'Target': np.random.randint(0, 2, n)
})

print("Sample Data with Categorical Variables:")
print(df_cat.head(10))

In [ ]:
# Method 1: Label Encoding
# Use for ORDINAL categories (with natural order)

df_encoded = df_cat.copy()

# Label Encoding for Size (has natural order)
le_size = LabelEncoder()
df_encoded['Size_Label'] = le_size.fit_transform(df_encoded['Size'])

print("Label Encoding for Size:")
print(df_encoded[['Size', 'Size_Label']].drop_duplicates().sort_values('Size_Label'))

print("\n⚠️ Problem: ML model thinks Medium (1) > Large (0)!")
print("   Use Ordinal Encoding instead for ordered categories")

In [ ]:
# Method 2: Ordinal Encoding
# Use for ORDINAL categories with specified order

# Define the order
size_order = ['Small', 'Medium', 'Large']
edu_order = ['High School', 'Bachelor', 'Master', 'PhD']

# Apply ordinal encoding
oe = OrdinalEncoder(categories=[size_order])
df_encoded['Size_Ordinal'] = oe.fit_transform(df_encoded[['Size']])

oe_edu = OrdinalEncoder(categories=[edu_order])
df_encoded['Education_Ordinal'] = oe_edu.fit_transform(df_encoded[['Education']])

print("Ordinal Encoding (with correct order):")
print("\nSize:")
print(df_encoded[['Size', 'Size_Ordinal']].drop_duplicates().sort_values('Size_Ordinal'))
print("\nEducation:")
print(df_encoded[['Education', 'Education_Ordinal']].drop_duplicates().sort_values('Education_Ordinal'))

In [ ]:
# Method 3: One-Hot Encoding
# Use for NOMINAL categories (no natural order)

# Using pandas get_dummies
df_onehot = pd.get_dummies(df_cat, columns=['Color', 'City'], prefix=['Color', 'City'])

print("One-Hot Encoding for Color and City:")
print(df_onehot.head(10))
print(f"\nOriginal columns: {df_cat.shape[1]}")
print(f"After One-Hot: {df_onehot.shape[1]}")

In [ ]:
# One-Hot Encoding with sklearn

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
color_encoded = ohe.fit_transform(df_cat[['Color']])

# Get feature names
feature_names = ohe.get_feature_names_out(['Color'])

# Create DataFrame
df_ohe = pd.DataFrame(color_encoded, columns=feature_names)

print("sklearn OneHotEncoder:")
print(df_ohe.head())

print("\nAdvantages of sklearn OneHotEncoder:")
print("- Can handle unknown categories")
print("- Easily integrated into pipelines")
print("- drop='first' to avoid multicollinearity")

In [ ]:
# Summary: When to use which encoding

encoding_guide = pd.DataFrame({
    'Encoding': ['Label Encoding', 'Ordinal Encoding', 'One-Hot Encoding'],
    'Use When': [
        'Binary categories (Yes/No)',
        'Ordered categories (Low/Medium/High)',
        'No natural order (Colors, Cities)'
    ],
    'Example': [
        'Gender: Male(0), Female(1)',
        'Size: Small(0), Medium(1), Large(2)',
        'City: [1,0,0], [0,1,0], [0,0,1]'
    ],
    'Pros': [
        'Simple, no extra columns',
        'Preserves order',
        'No artificial ordering'
    ],
    'Cons': [
        'Implies false ordering',
        'Needs manual order definition',
        'Many columns for high cardinality'
    ]
})

print("=== Encoding Guide ===")
print(encoding_guide.to_string(index=False))

---

# Part D: Feature Scaling

## 4.5 Different Scaling Techniques

In [ ]:
# Create sample data with different scales

np.random.seed(42)
n = 200

df_scale = pd.DataFrame({
    'Age': np.random.randint(18, 65, n),           # Range: 18-65
    'Income': np.random.normal(55000, 20000, n),    # Range: ~15000-95000
    'Credit_Score': np.random.randint(300, 850, n), # Range: 300-850
    'Purchases': np.random.poisson(10, n)          # Range: 0-25
})

# Add some outliers to Income
df_scale.loc[0, 'Income'] = 200000
df_scale.loc[1, 'Income'] = 250000

print("Original Data Statistics:")
print(df_scale.describe().round(2))

In [ ]:
# Visualize different scales

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, df_scale.columns):
    ax.hist(df_scale[col], bins=20, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}\nRange: {df_scale[col].min():.0f} - {df_scale[col].max():.0f}', 
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Features at Different Scales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n⚠️ Problem: Features have very different ranges!")
print("   Income dominates distance-based algorithms (KNN, SVM, etc.)")

In [ ]:
# Scaling Method 1: StandardScaler (Z-score normalization)
# Formula: z = (x - mean) / std
# Result: mean=0, std=1

standard_scaler = StandardScaler()
df_standard = pd.DataFrame(
    standard_scaler.fit_transform(df_scale),
    columns=df_scale.columns
)

print("StandardScaler Result:")
print(df_standard.describe().round(2))

print("\n✅ Use StandardScaler when:")
print("   - Data is normally distributed")
print("   - Using algorithms sensitive to magnitude")

In [ ]:
# Scaling Method 2: MinMaxScaler (Normalization)
# Formula: x_scaled = (x - min) / (max - min)
# Result: values between 0 and 1

minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df_scale),
    columns=df_scale.columns
)

print("MinMaxScaler Result:")
print(df_minmax.describe().round(2))

print("\n✅ Use MinMaxScaler when:")
print("   - You need values in [0, 1] range")
print("   - Using neural networks with sigmoid")
print("\n⚠️ Sensitive to outliers!")

In [ ]:
# Scaling Method 3: RobustScaler
# Formula: x_scaled = (x - median) / IQR
# Result: Robust to outliers

robust_scaler = RobustScaler()
df_robust = pd.DataFrame(
    robust_scaler.fit_transform(df_scale),
    columns=df_scale.columns
)

print("RobustScaler Result:")
print(df_robust.describe().round(2))

print("\n✅ Use RobustScaler when:")
print("   - Data has outliers")
print("   - Want to preserve outliers but reduce their impact")

In [ ]:
# Compare all scalers

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original Income distribution
axes[0, 0].hist(df_scale['Income'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Original Income', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Income')
axes[0, 0].grid(True, alpha=0.3)

# StandardScaler
axes[0, 1].hist(df_standard['Income'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('StandardScaler', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Scaled Income')
axes[0, 1].grid(True, alpha=0.3)

# MinMaxScaler
axes[1, 0].hist(df_minmax['Income'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_title('MinMaxScaler', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Scaled Income')
axes[1, 0].grid(True, alpha=0.3)

# RobustScaler
axes[1, 1].hist(df_robust['Income'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title('RobustScaler', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Scaled Income')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Comparison of Scaling Methods on Income (with outliers)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

# Part E: Feature Creation

## 4.6 Creating New Features

In [ ]:
# Sample dataset for feature creation

np.random.seed(42)
n = 200

df_features = pd.DataFrame({
    'Date': pd.date_range('2023-01-01', periods=n, freq='D'),
    'Price': np.random.uniform(100, 500, n),
    'Quantity': np.random.randint(1, 50, n),
    'Width': np.random.uniform(10, 100, n),
    'Height': np.random.uniform(10, 100, n),
    'Customer_Age': np.random.randint(18, 70, n),
    'Years_Customer': np.random.randint(0, 20, n)
})

print("Original Data:")
print(df_features.head(10))

In [ ]:
# Feature Engineering: Creating new features

df_eng = df_features.copy()

# 1. Arithmetic combinations
df_eng['Total_Value'] = df_eng['Price'] * df_eng['Quantity']
df_eng['Area'] = df_eng['Width'] * df_eng['Height']
df_eng['Price_Per_Unit_Area'] = df_eng['Price'] / df_eng['Area']

print("1. Arithmetic Combinations:")
print(df_eng[['Price', 'Quantity', 'Total_Value', 'Area']].head())

# 2. Date features
df_eng['Year'] = df_eng['Date'].dt.year
df_eng['Month'] = df_eng['Date'].dt.month
df_eng['Day'] = df_eng['Date'].dt.day
df_eng['DayOfWeek'] = df_eng['Date'].dt.dayofweek
df_eng['IsWeekend'] = df_eng['DayOfWeek'].isin([5, 6]).astype(int)
df_eng['Quarter'] = df_eng['Date'].dt.quarter

print("\n2. Date Features:")
print(df_eng[['Date', 'Month', 'DayOfWeek', 'IsWeekend', 'Quarter']].head())

# 3. Binning (Age groups)
df_eng['Age_Group'] = pd.cut(df_eng['Customer_Age'], 
                              bins=[0, 25, 35, 50, 100],
                              labels=['Young', 'Adult', 'Middle', 'Senior'])

print("\n3. Binning (Age Groups):")
print(df_eng[['Customer_Age', 'Age_Group']].head(10))

In [ ]:
# 4. Polynomial Features

from sklearn.preprocessing import PolynomialFeatures

# Select numerical features
X_poly = df_features[['Price', 'Quantity']].head(10)

# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_transformed = poly.fit_transform(X_poly)

# Get feature names
feature_names = poly.get_feature_names_out(['Price', 'Quantity'])

df_poly = pd.DataFrame(X_poly_transformed, columns=feature_names)

print("Polynomial Features (degree=2):")
print(df_poly.round(2))

print("\nNew features created:")
print(f"- {feature_names}")

In [ ]:
# 5. Aggregation features (for grouped data)

# Simulate customer transactions
np.random.seed(42)
transactions = pd.DataFrame({
    'Customer_ID': np.random.randint(1, 20, 100),
    'Amount': np.random.uniform(10, 500, 100),
    'Category': np.random.choice(['Food', 'Electronics', 'Clothing'], 100)
})

# Aggregate by customer
customer_features = transactions.groupby('Customer_ID').agg({
    'Amount': ['sum', 'mean', 'max', 'min', 'count', 'std']
}).round(2)

# Flatten column names
customer_features.columns = ['Total_Spent', 'Avg_Transaction', 'Max_Transaction', 
                              'Min_Transaction', 'Num_Transactions', 'Std_Transaction']

print("Aggregation Features by Customer:")
print(customer_features.head(10))

---

# Part F: Feature Selection

## 4.7 Selecting Important Features

In [ ]:
# Create dataset with many features

from sklearn.datasets import make_classification

# Generate data with some informative and some noise features
X, y = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_classes=2,
    random_state=42
)

# Create DataFrame
feature_names = [f'Feature_{i}' for i in range(20)]
df_select = pd.DataFrame(X, columns=feature_names)
df_select['Target'] = y

print("Dataset for Feature Selection:")
print(f"Shape: {df_select.shape}")
print(f"\n8 informative, 4 redundant, 8 noise features")

In [ ]:
# Method 1: Correlation Analysis

# Calculate correlations with target
correlations = df_select.corr()['Target'].drop('Target').abs().sort_values(ascending=False)

print("Feature Correlations with Target:")
print(correlations.round(3))

# Visualize
plt.figure(figsize=(12, 6))
correlations.plot(kind='bar', color='steelblue', edgecolor='black')
plt.axhline(y=0.1, color='red', linestyle='--', label='Threshold (0.1)')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Absolute Correlation', fontsize=12)
plt.title('Feature Correlation with Target', fontsize=14, fontweight='bold')
plt.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: SelectKBest (Statistical Tests)

X = df_select.drop('Target', axis=1)
y = df_select['Target']

# Using F-test
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)

# Get selected features
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()
scores = selector.scores_

print("SelectKBest (top 10 features):")
print(f"Selected: {selected_features}")

# Visualize scores
plt.figure(figsize=(12, 6))
plt.bar(X.columns, scores, color=['green' if s else 'gray' for s in selected_mask], edgecolor='black')
plt.xlabel('Features', fontsize=12)
plt.ylabel('F-Score', fontsize=12)
plt.title('SelectKBest Scores (Green = Selected)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Method 3: Feature Importance from Tree-Based Models

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Get feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("Random Forest Feature Importances:")
print(importances.round(3))

# Visualize
plt.figure(figsize=(12, 6))
importances.plot(kind='bar', color='forestgreen', edgecolor='black')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Importance', fontsize=12)
plt.title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Method 4: Recursive Feature Elimination (RFE)

from sklearn.feature_selection import RFE

# Use Logistic Regression as estimator
estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator, n_features_to_select=10, step=1)
rfe.fit(X, y)

# Get rankings
rankings = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("RFE Rankings (1 = best):")
print(rankings)

rfe_selected = rankings[rankings['Selected']]['Feature'].tolist()
print(f"\nSelected Features: {rfe_selected}")

In [ ]:
# Compare Feature Selection Methods

# Combine results
comparison = pd.DataFrame({
    'Feature': X.columns,
    'Correlation': df_select.corr()['Target'].drop('Target').abs()[X.columns].values,
    'SelectKBest_Score': selector.scores_,
    'RF_Importance': rf.feature_importances_,
    'RFE_Rank': rfe.ranking_
})

# Normalize scores for comparison
for col in ['Correlation', 'SelectKBest_Score', 'RF_Importance']:
    comparison[f'{col}_Norm'] = (comparison[col] - comparison[col].min()) / (comparison[col].max() - comparison[col].min())

comparison['RFE_Rank_Norm'] = 1 - (comparison['RFE_Rank'] - 1) / (comparison['RFE_Rank'].max() - 1)

# Calculate average score
comparison['Avg_Score'] = comparison[['Correlation_Norm', 'SelectKBest_Score_Norm', 
                                      'RF_Importance_Norm', 'RFE_Rank_Norm']].mean(axis=1)

comparison = comparison.sort_values('Avg_Score', ascending=False)

print("=== Feature Selection Comparison ===")
print(comparison[['Feature', 'Correlation', 'SelectKBest_Score', 'RF_Importance', 
                  'RFE_Rank', 'Avg_Score']].round(3))

---

## 📝 Practice Exercises

### Exercise 1: Complete Data Preprocessing Pipeline

In [ ]:
# Exercise: Create a complete preprocessing pipeline

# Generate messy dataset
np.random.seed(42)
n = 300

messy_data = pd.DataFrame({
    'Age': np.random.randint(18, 70, n),
    'Income': np.random.normal(50000, 20000, n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'City': np.random.choice(['NYC', 'LA', 'Chicago'], n),
    'Experience': np.random.randint(0, 40, n),
    'Target': np.random.randint(0, 2, n)
})

# Add missing values
messy_data.loc[np.random.choice(n, 20), 'Age'] = np.nan
messy_data.loc[np.random.choice(n, 15), 'Income'] = np.nan
messy_data.loc[np.random.choice(n, 10), 'Education'] = np.nan

# Add outliers
messy_data.loc[0, 'Income'] = 500000
messy_data.loc[1, 'Income'] = 600000

print("Messy Dataset:")
print(messy_data.head(10))
print(f"\nMissing values: {messy_data.isnull().sum().sum()}")

# TODO:
# 1. Handle missing values (numeric: median, categorical: mode)
# 2. Detect and handle outliers in Income
# 3. Encode categorical variables
# 4. Scale numerical features
# 5. Create new features (e.g., Income_per_Experience)

print("\nComplete the preprocessing pipeline!")

---

## ✅ Phase Completion Checklist

- [ ] Handle missing data with different strategies
- [ ] Detect and treat outliers using IQR and Z-score
- [ ] Apply Label, Ordinal, and One-Hot encoding
- [ ] Use StandardScaler, MinMaxScaler, and RobustScaler
- [ ] Create new features from existing data
- [ ] Select important features using multiple methods

---

## 🎯 Key Takeaways

1. **Missing Data**: Choose strategy based on data type and amount
2. **Outliers**: Context matters - sometimes outliers are valuable information
3. **Encoding**: Match encoding type to category type (ordinal vs nominal)
4. **Scaling**: Use RobustScaler for data with outliers
5. **Feature Creation**: Domain knowledge helps create meaningful features
6. **Feature Selection**: Use multiple methods and compare results

---

## 📚 Next Steps

👉 **[Phase-5-6-Advanced-ML.ipynb](Phase-5-6-Advanced-ML.ipynb)** - Advanced techniques!

Master:
- Cross-validation strategies
- Hyperparameter tuning (GridSearch, RandomSearch)
- Ensemble methods (Bagging, Boosting)
- XGBoost and LightGBM

**Happy Learning! 🚀**